In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

<font size="5" color="red">lstm</font>
# 영화평 분석

In [2]:
#1. 패키지
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from time import time # 70.1.1부터 현재까지의 밀리세컨
    
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

In [3]:
# 하이퍼 파리미터 설정(이 파라미터를 바꾸면 정확도나 학습속도에 차이남)
MY_WORDS = 10000 # imdb 대이터의 단어수
MY_LENGTH = 80  # 영화평 단어수 80개만 독립변수
MY_EMBED = 32 # Embeding layer 의 결과 차원
MY_HIDDEN = 64
MY_EPOCH = 10  # 학습 수
MY_BATCH = 200 # batch_size(fit시 매번 데이터를 가져오는 데이터)

In [4]:
# 데이터 불러오기
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MY_WORDS)

In [5]:
print('학습셋 입력수 모양 :',x_train.shape)
print('학습셋 타겟변수 모양 :', y_train.shape)
print(x_train[2], y_train[2])
print(x_test.shape, y_test.shape)

학습셋 입력수 모양 : (25000,)
학습셋 타겟변수 모양 : (25000,)
[1, 14, 47, 8, 30, 31, 7, 4, 249, 108, 7, 4, 5974, 54, 61, 369, 13, 71, 149, 14, 22, 112, 4, 2401, 311, 12, 16, 3711, 33, 75, 43, 1829, 296, 4, 86, 320, 35, 534, 19, 263, 4821, 1301, 4, 1873, 33, 89, 78, 12, 66, 16, 4, 360, 7, 4, 58, 316, 334, 11, 4, 1716, 43, 645, 662, 8, 257, 85, 1200, 42, 1228, 2578, 83, 68, 3912, 15, 36, 165, 1539, 278, 36, 69, 2, 780, 8, 106, 14, 6905, 1338, 18, 6, 22, 12, 215, 28, 610, 40, 6, 87, 326, 23, 2300, 21, 23, 22, 12, 272, 40, 57, 31, 11, 4, 22, 47, 6, 2307, 51, 9, 170, 23, 595, 116, 595, 1352, 13, 191, 79, 638, 89, 2, 14, 9, 8, 106, 607, 624, 35, 534, 6, 227, 7, 129, 113] 0
(25000,) (25000,)


In [6]:
# 긍정 / 부정 갯수
print('학습셋의 긍정 갯수:',y_train.sum())
print('테스트셋의 긍정 갯수', y_test.sum())

학습셋의 긍정 갯수: 12500
테스트셋의 긍정 갯수 12500


In [7]:
# 긍정 / 부정 갯수
print(y_train.sum())

12500


# 문자단어 -> 정수

In [8]:
word_to_id = imdb.get_word_index()
print(word_to_id['movie'])
print(word_to_id['film'])
print(word_to_id['a'])
print(word_to_id['the'])
id_to_word ={}
for word, value in word_to_id.items():
    id_to_word[value] = word
print(id_to_word[1])
    

17
19
3
1
the


In [9]:
word_to_id

{'fawn': 34701,
 'tsukino': 52006,
 'nunnery': 52007,
 'sonja': 16816,
 'vani': 63951,
 'woods': 1408,
 'spiders': 16115,
 'hanging': 2345,
 'woody': 2289,
 'trawling': 52008,
 "hold's": 52009,
 'comically': 11307,
 'localized': 40830,
 'disobeying': 30568,
 "'royale": 52010,
 "harpo's": 40831,
 'canet': 52011,
 'aileen': 19313,
 'acurately': 52012,
 "diplomat's": 52013,
 'rickman': 25242,
 'arranged': 6746,
 'rumbustious': 52014,
 'familiarness': 52015,
 "spider'": 52016,
 'hahahah': 68804,
 "wood'": 52017,
 'transvestism': 40833,
 "hangin'": 34702,
 'bringing': 2338,
 'seamier': 40834,
 'wooded': 34703,
 'bravora': 52018,
 'grueling': 16817,
 'wooden': 1636,
 'wednesday': 16818,
 "'prix": 52019,
 'altagracia': 34704,
 'circuitry': 52020,
 'crotch': 11585,
 'busybody': 57766,
 "tart'n'tangy": 52021,
 'burgade': 14129,
 'thrace': 52023,
 "tom's": 11038,
 'snuggles': 52025,
 'francesco': 29114,
 'complainers': 52027,
 'templarios': 52125,
 '272': 40835,
 '273': 52028,
 'zaniacs': 52130,

In [10]:
msg = 'What a wonderful movie'
msg = msg.lower().split()
msg
#1 : 리뷰 시작을 알리는 숫자, 2: 문자가 짤려서 잘못 읽어옴
data= [1] + [word_to_id.get(m, -1)+3 for m in msg]
print(msg)
word_to_id['what'] +3
print([id_to_word.get(d-3,'???') for d in data])

['what', 'a', 'wonderful', 'movie']
['???', 'what', 'a', 'wonderful', 'movie']


# 5. 숫자영화평 -> 자연어 영화평 return  함수

In [11]:
def decoding(review_num):
    decoded = [id_to_word.get(num-3, '???') for num in review_num]
    return ' '.join(decoded)


In [12]:
decoding(x_train[5])

"??? begins better than it ends funny that the russian submarine crew ??? all other actors it's like those scenes where documentary shots br br spoiler part the message ??? was contrary to the whole story it just does not ??? br br"

# 6. 영화평(입력변수)의 길이


In [13]:
def show_length(x_train):
    print('첫 20개 영화평 길이')
    print([len(x_data) for x_data in x_train[:20]])

In [14]:
show_length(x_train)

첫 20개 영화평 길이
[218, 189, 141, 550, 147, 43, 123, 562, 233, 130, 450, 99, 117, 238, 109, 129, 163, 752, 212, 177]


In [15]:
print('제일 긴 영화평 단어 길이:',
     max(len(x_data) for x_data in x_train))
print('제일 짧은 영화평 단어 길이:',
     min(len(x_data) for x_data in x_train))

제일 긴 영화평 단어 길이: 2494
제일 짧은 영화평 단어 길이: 11


# 7. 모든 영화평 길이를 동일하게


In [16]:
X_train = pad_sequences(x_train,
                        padding='post',
                        truncating='pre',   # 뒷부분을 짜르고 앞부분을 남김
                        maxlen=MY_LENGTH)
X_test = pad_sequences(x_test,
                       padding='post',
                       truncating='pre',
                       maxlen=MY_LENGTH)


show_length(X_train)  # X_train의 길이를 출력
show_length(X_test)   # X_test의 길이를 출력


첫 20개 영화평 길이
[80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80]
첫 20개 영화평 길이
[80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80]


In [17]:
X_train.shape

(25000, 80)

# 9. 모델생성

In [18]:
model = Sequential()
model.add(Embedding(input_dim=MY_WORDS,
                   output_dim=MY_EMBED,
                   input_length=MY_LENGTH))
#RNN : 입력 단어의 길이수가 너무 길면 파라미터 없데이트가 안됨
model.add(LSTM(units=MY_HIDDEN,
              input_shape=(MY_LENGTH, MY_EMBED)))
model.add(Dense(units=1, activation='sigmoid'))
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 80, 32)            320000    
                                                                 
 lstm (LSTM)                 (None, 64)                24832     
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 344,897
Trainable params: 344,897
Non-trainable params: 0
_________________________________________________________________


# 학습환경 설정 및 학습하기

In [19]:
%%time
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['acc'])  # 이진 분류 시 손실 함수
hist = model.fit(X_train, y_train,
                 epochs=MY_EPOCH,  # 'eopchs' -> 'epochs'
                 batch_size=MY_BATCH,
                 validation_split=0.2,
                 verbose=1)


Epoch 1/10
100/100 [==============================] - 11s 94ms/step - loss: 0.5405 - acc: 0.7064 - val_loss: 0.3946 - val_acc: 0.8152
Epoch 2/10
100/100 [==============================] - 9s 90ms/step - loss: 0.3165 - acc: 0.8698 - val_loss: 0.3675 - val_acc: 0.8374
Epoch 3/10
100/100 [==============================] - 9s 89ms/step - loss: 0.2435 - acc: 0.9090 - val_loss: 0.3743 - val_acc: 0.8296
Epoch 4/10
100/100 [==============================] - 9s 88ms/step - loss: 0.2012 - acc: 0.9277 - val_loss: 0.4663 - val_acc: 0.8222
Epoch 5/10
100/100 [==============================] - 9s 88ms/step - loss: 0.1679 - acc: 0.9408 - val_loss: 0.4695 - val_acc: 0.8160
Epoch 6/10
100/100 [==============================] - 9s 87ms/step - loss: 0.1512 - acc: 0.9471 - val_loss: 0.5043 - val_acc: 0.8122
Epoch 7/10
100/100 [==============================] - 9s 88ms/step - loss: 0.1286 - acc: 0.9580 - val_loss: 0.5714 - val_acc: 0.8114
Epoch 8/10
100/100 [==============================] - 9s 88ms/step -

In [20]:
# 혼동행렬
yhat = model.predict(X_test)
yhat

782/782 [==============================] - 9s 11ms/step


array([[0.02004693],
       [0.99811715],
       [0.8072532 ],
       ...,
       [0.00488027],
       [0.0035776 ],
       [0.9642158 ]], dtype=float32)

In [23]:
yhat=(model.predict(X_test)>0.5).astype(np.int16).reshape(-1)
yhat

782/782 [==============================] - 9s 12ms/step


array([0, 1, 1, ..., 0, 0, 1], dtype=int16)

In [24]:
y_test

array([0, 1, 1, ..., 0, 0, 0], dtype=int64)

In [25]:
confusion_matrix(y_test, yhat)

array([[10113,  2387],
       [ 2678,  9822]], dtype=int64)

In [27]:
recall_score(y_test, yhat)

0.78576

In [28]:
# precision (True로 예측한 것 중 실제값이 True 인 비율) 10238/(2778+10238)
precision_score(y_test, yhat)

0.8044884920959947

## 12.모델 사용하기 review

In [29]:
review = ' the movie is quite interesting and i cant stop watch clock during running time. in the sreen all hereos gathered togeher, akd it give me goodsbumb. Also i appreicate to mavel studio how the end storyline of decade in emotional way.'

In [30]:
review

' the movie is quite interesting and i cant stop watch clock during running time. in the sreen all hereos gathered togeher, akd it give me goodsbumb. Also i appreicate to mavel studio how the end storyline of decade in emotional way.'

In [35]:
import re
review = re.sub('[^a-zA-Z\'\s]','',review)

review = review.split()
review

['the',
 'movie',
 'is',
 'quite',
 'interesting',
 'and',
 'i',
 'cant',
 'stop',
 'watch',
 'clock',
 'during',
 'running',
 'time',
 'in',
 'the',
 'sreen',
 'all',
 'hereos',
 'gathered',
 'togeher',
 'akd',
 'it',
 'give',
 'me',
 'goodsbumb',
 'Also',
 'i',
 'appreicate',
 'to',
 'mavel',
 'studio',
 'how',
 'the',
 'end',
 'storyline',
 'of',
 'decade',
 'in',
 'emotional',
 'way']

In [49]:
review = 'the movie is quite interesting and i cant stop watch clock during running time. in the sreen all hereos gathered togeher, akd it give me goodsbump. Also i appreicate to mavel studio how the end storyline of decade in emotional way.'
import re

# 특수 문자 제거
review = re.sub('[^a-zA-Z\'\s]','', review)

# 단어 리스트로 나누기
review = review.split()

# 임베딩 적용
review = [1] + [word_to_id.get(word, -1) + 3 for word in review]

# 2차원 배열로 변환
review_2d = [review]  # 2차원 배열, review 리스트를 하나의 행으로

# 출력
print(review_2d)


[[1, 4, 20, 9, 179, 221, 5, 13, 2488, 570, 106, 5431, 315, 620, 58, 11, 4, 2, 32, 2, 8676, 2, 2, 12, 202, 72, 2, 2, 13, 2, 8, 2, 1182, 89, 4, 130, 769, 7, 2068, 11, 921, 96]]


In [50]:
review = pad_sequences(review,
                       padding='post',
                       truncating='pre',
                       maxlen=MY_LENGTH)

ValueError: `sequences` must be a list of iterables. Found non-iterable: 1